# 02 · Medallion pipeline: Turso → Bronze → Silver → Gold
| Step | What | Where |
|---|---|---|
| 1 | **Stage**: incremental extract from Turso, Parquet landing, append to Bronze as text | `bronze.*` |
| 2 | **Clean**: cast, standardise, DQ rules → quarantine, dedupe, guarded merge | `silver.*` |
| 3 | **Join**: sales ⟕ conformed dimensions, surrogate keys, Unknown member | in-flight |
| 4 | **Merge** into the Gold dimensions and fact | `gold.*` |
| 5 | **Delta** (optional): only Silver rows changed since the last Gold load | `ctl.watermark` |

Run it after each source batch (notebook 01). `LOAD_MODE=full` rebuilds Gold from all of Silver.

In [1]:
%run -i ./00_config.ipynb

In [2]:
open_warehouse()
src = source_conn()
new_batch()
print(f"run batch {BATCH_ID} | gold load mode: {LOAD_MODE} | source: {'Turso' if TURSO_URL else 'local SQLite stand-in'}")

run batch 1 | gold load mode: incremental | source: Turso


## 1 · Stage (Bronze)

In [3]:
for e in ENTITIES: stage(e)

  customers  extracted    60 rows (since the beginning),    60 new in bronze
  products   extracted    20 rows (since the beginning),    20 new in bronze
  sales      extracted   304 rows (since the beginning),   304 new in bronze


## 2 · Clean (Silver)

In [4]:
for e in ENTITIES: clean_to_silver(e)

  customers     60 valid latest rows ->    60 inserted/updated,   0 quarantined
  products      19 valid latest rows ->    19 inserted/updated,   1 quarantined
  sales        300 valid latest rows ->   300 inserted/updated,   4 quarantined


## 3 · Join
Dimensions first (so keys exist), then join the sales delta to them.

In [5]:
for d in DIMS: load_dim(d)
print(f"  fact join   {build_fact_join():5d} rows ready (delta + any facts waiting on a late dimension)")

  dim_customer     60 inserted/updated
  dim_product      19 inserted/updated
  fact join     300 rows ready (delta + any facts waiting on a late dimension)


## 4 · Merge into Gold

In [6]:
print(f"  fact_sales  {merge_fact():5d} inserted/updated")

  fact_sales    300 inserted/updated


## 5 · Delta, watermarks & checks
Run the pipeline again with no new source data and every step reports "nothing new".

In [7]:
display(con.execute("SELECT name, value, updated_ts FROM ctl.watermark ORDER BY name").df())
display(con.execute("""
  SELECT 'bronze' AS layer, 'sales' AS tbl, count(*) AS n FROM bronze.sales UNION ALL
  SELECT 'silver', 'sales', count(*) FROM silver.sales UNION ALL
  SELECT 'silver', '_quarantine (all entities)', count(*) FROM silver._quarantine UNION ALL
  SELECT 'gold', 'fact_sales', count(*) FROM gold.fact_sales UNION ALL
  SELECT 'gold', 'fact rows with unknown dim key', count(*) FROM gold.fact_sales WHERE customer_sk = -1 OR product_sk = -1
""").df())

,name,value,updated_ts
0,extract.customers,2026-10-30 14:13:55,2026-11-18 17:24:35.489720
1,extract.products,2026-10-31 18:47:42,2026-11-18 17:24:35.513005
2,extract.sales,2026-10-31 20:05:30,2026-11-18 17:24:35.537825
3,gold.dim_customer,1,2026-11-18 17:24:37.839766
4,gold.dim_product,1,2026-11-18 17:24:37.864092
5,gold.fact_sales,1,2026-11-18 17:24:38.162025
6,silver.customers,1,2026-11-18 17:24:36.795456
7,silver.products,1,2026-11-18 17:24:36.822644
8,silver.sales,1,2026-11-18 17:24:36.850593


,layer,tbl,n
0,bronze,sales,304
1,silver,sales,300
2,silver,_quarantine (all entities),5
3,gold,fact_sales,300
4,gold,fact rows with unknown dim key,19


In [8]:
display(con.execute("SELECT entity, batch_id, payload FROM silver._quarantine ORDER BY batch_id, entity LIMIT 10").df())
display(con.execute("""
  SELECT f.order_date, c.name, c.segment, p.product_name, f.quantity, f.amount
  FROM gold.fact_sales f JOIN gold.dim_customer c USING (customer_sk) JOIN gold.dim_product p USING (product_sk)
  ORDER BY f.order_ts DESC LIMIT 10""").df())

,entity,batch_id,payload
0,products,1,"{""product_id"":""19"",""product_name"":""Webcam"",""ca..."
1,sales,1,"{""order_line_id"":""10091"",""order_id"":""5034"",""cu..."
2,sales,1,"{""order_line_id"":""10008"",""order_id"":""5003"",""cu..."
3,sales,1,"{""order_line_id"":""10042"",""order_id"":""5016"",""cu..."
4,sales,1,"{""order_line_id"":""10131"",""order_id"":""5048"",""cu..."


,order_date,name,segment,product_name,quantity,amount
0,2026-10-31,Jamal Silva,smb,Cable Pack,5,37.50
1,2026-10-31,Hector Brown,smb,Pro Kit,3,161.97
2,2026-10-31,Hector Brown,smb,Starter Kit,1,21.99
3,2026-10-31,Uma Garcia,retail,Cloud Seat,1,12.00
4,2026-10-31,Uma Nguyen,retail,Docking Station,3,348.30
5,2026-10-31,Bob Lee,enterprise,Analytics Add-on,3,161.70
6,2026-10-31,Hector Brown,smb,Backup Plan,4,60.00
7,2026-10-31,Uma Nguyen,retail,Gizmo,4,18.00
8,2026-10-31,Uma Garcia,retail,Widget,2,19.98
9,2026-10-31,Uma Nguyen,retail,Backup Plan,3,45.00


## Scaling notes
* **Source volume:** extraction is keyset-style (`updated_at` watermark, ordered) in `CHUNK`-row Parquet parts, so memory stays flat. Use `TURSO_REMOTE=1` to avoid holding a local replica of a very large source.
* **Watermark caveat:** the extract assumes `updated_at` is set on every insert and update. Deletes are not seen; add an `is_deleted` flag or a tombstone table if you need them.
* **Warehouse size:** DuckDB comfortably handles hundreds of millions of rows on a Codespace; for more, point the same SQL at MotherDuck or write the layers as Parquet/Delta on object storage.
* **Orchestration:** the five steps are plain functions; call them from GitHub Actions on a schedule (`jupyter nbconvert --execute`) or from any scheduler.
* **Porting back to Databricks:** the metadata, step order and watermark logic carry over unchanged; the helpers become Auto Loader, Delta `MERGE` and Change Data Feed.

In [11]:
con.sql("SELECT * FROM ctl.watermark ORDER BY name").show()
con.sql("SELECT * FROM ctl.batch_log").show()

┌───────────────────┬─────────────────────┬────────────────────────────┐
│       name        │        value        │         updated_ts         │
│      varchar      │       varchar       │         timestamp          │
├───────────────────┼─────────────────────┼────────────────────────────┤
│ extract.customers │ 2026-10-30 14:13:55 │ 2026-11-18 17:24:35.48972  │
│ extract.products  │ 2026-10-31 18:47:42 │ 2026-11-18 17:24:35.513005 │
│ extract.sales     │ 2026-10-31 20:05:30 │ 2026-11-18 17:24:35.537825 │
│ gold.dim_customer │ 1                   │ 2026-11-18 17:24:37.839766 │
│ gold.dim_product  │ 1                   │ 2026-11-18 17:24:37.864092 │
│ gold.fact_sales   │ 1                   │ 2026-11-18 17:24:38.162025 │
│ silver.customers  │ 1                   │ 2026-11-18 17:24:36.795456 │
│ silver.products   │ 1                   │ 2026-11-18 17:24:36.822644 │
│ silver.sales      │ 1                   │ 2026-11-18 17:24:36.850593 │
└───────────────────┴─────────────────────┴────────